# One Function

In [1]:
import os
import joblib
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sqlalchemy import create_engine
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk

nltk.download(['punkt', 'wordnet'])

# Function to tokenize text
def tokenize(text):
    tokens = word_tokenize(text)
    lemmatizer = WordNetLemmatizer()

    clean_tokens = [lemmatizer.lemmatize(tok).lower().strip() for tok in tokens]
    return clean_tokens

# Function to change directory
def change_directory(directory):
    original_directory = os.getcwd()
    os.chdir(directory)
    return original_directory

# Load data from database
def load_data(database_filepath):
    engine = create_engine(f'sqlite:///{database_filepath}')
    with engine.connect() as connection:
        df = pd.read_sql("SELECT * FROM messages", connection)
    engine.dispose()
    return df

# Function to balance dataset
def balance_data(df, target_column):
    df_r = df[['message', target_column]]
    df_1 = df_r[df_r[target_column] == 1]
    df_0 = df_r[df_r[target_column] == 0]

    random_seed = 42
    df_1_balanced = df_1.sample(len(df_0), replace=True, random_state=random_seed)
    df_balanced = pd.concat([df_0, df_1_balanced]).sample(frac=1, random_state=random_seed).reset_index(drop=True)
    return df_balanced

# Main training function
def main():
    dataset_directory = './dataset'
    database_filepath = 'DisasterResponse.db'
    target_column = 'related'

    # Change to dataset directory
    original_directory = change_directory(dataset_directory)

    try:
        df = load_data(database_filepath)
    finally:
        os.chdir(original_directory)

    df_balanced = balance_data(df, target_column)
    X = df_balanced['message']
    Y = df_balanced[target_column]

    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)

    pipeline = Pipeline([
        ('vect', CountVectorizer(tokenizer=tokenize, token_pattern=None, ngram_range=(1, 3))),
        ('tfidf', TfidfTransformer()),
        ('clf', RandomForestClassifier())
    ])

    param_grid = [
        {
            'clf': [RandomForestClassifier()],
            'clf__n_estimators': [100, 200],
            'clf__min_samples_split': [2, 5]
        },
        {
            'clf': [LogisticRegression(max_iter=1000)],
            'clf__C': [0.1, 1, 10],
            'clf__solver': ['liblinear', 'saga']
        },
        {
            'clf': [GradientBoostingClassifier()],
            'clf__n_estimators': [100, 200],
            'clf__learning_rate': [0.01, 0.1, 0.2]
        }
    ]

    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='precision_weighted', n_jobs=1, verbose=2)
    grid_search.fit(X_train, Y_train)

    # Save the best model
    joblib.dump(grid_search.best_estimator_, 'best_mode_related.pkl')

    # Evaluate the best model
    Y_pred = grid_search.best_estimator_.predict(X_test)
    precision = precision_score(Y_test, Y_pred, average='weighted', zero_division=0)
    recall = recall_score(Y_test, Y_pred, average='weighted', zero_division=0)
    f1 = f1_score(Y_test, Y_pred, average='weighted', zero_division=0)
    overall_accuracy = accuracy_score(Y_test, Y_pred)

    print(f'Overall Accuracy: {overall_accuracy:.4f}')
    print(f'Macro Average Precision: {precision:.4f}')
    print(f'Macro Average Recall: {recall:.4f}')
    print(f'Macro Average F1 Score: {f1:.4f}')

if __name__ == "__main__":
    main()


c:\Users\sinde\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\sinde\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Fitting 3 folds for each of 16 candidates, totalling 48 fits
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=100; total time=  31.0s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=100; total time=  36.6s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=100; total time=  37.0s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=200; total time= 1.3min
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=200; total time= 1.2min
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=2, clf__n_estimators=200; total time= 1.2min
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=5, clf__n_estimators=100; total time=  28.4s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=5, clf__n_estimators=100; total time=  31.0s
[CV] END clf=RandomForestClassifier(), clf__min_samples_split=5, clf__n_estimators=